# Tutorial: CNN 03 Transfer Learning

Audience:
- Learners who want to see how a pretrained convolutional backbone can be reused on a related task.

Prerequisites:
- The previous CNN classification notebook and the discussion of transfer learning in the module notes.

Learning goals:
- Pretrain a CNN backbone on one real image task.
- Reuse that backbone on a new target task with limited labels.
- Compare frozen-feature transfer, fine-tuning, and training from scratch.


## Outline

1. Build a source task from the digits dataset.
2. Pretrain a backbone on 10-way digit classification.
3. Transfer to a smaller prime-vs-non-prime classification task.
4. Compare frozen transfer, fine-tuning, and random initialization.


In [ ]:
from __future__ import annotations

import copy
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 21
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
X = X[:, None, :, :]
y_source = digits.target.astype(np.int64)
prime_digits = {2, 3, 5, 7}
y_target = np.array([1 if label in prime_digits else 0 for label in y_source], dtype=np.int64)

X_source_train, X_source_test, y_source_train, y_source_test, y_target_train, y_target_test = train_test_split(
    X, y_source, y_target, test_size=0.2, random_state=SEED, stratify=y_source
)

# Keep the target task label-efficient on purpose.
small_target_indices = []
for cls in [0, 1]:
    cls_idx = np.where(y_target_train == cls)[0][:120]
    small_target_indices.extend(cls_idx.tolist())
small_target_indices = np.array(small_target_indices)

X_target_small = X_source_train[small_target_indices]
y_target_small = y_target_train[small_target_indices]

X_source_train.shape, X_target_small.shape


## Step 1 - Define a reusable backbone

We separate the feature extractor from the classification head so the same convolutional representation can support multiple tasks.


In [ ]:
class ConvBackbone(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class CNNHead(nn.Module):
    def __init__(self, num_classes: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class FullModel(nn.Module):
    def __init__(self, backbone: ConvBackbone, num_classes: int) -> None:
        super().__init__()
        self.backbone = backbone
        self.head = CNNHead(num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))


In [ ]:
@dataclass
class RunHistory:
    train_loss: list[float]
    val_accuracy: list[float]


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int = 64, shuffle: bool = False) -> DataLoader:
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def predict(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            preds.append(logits.argmax(dim=1).cpu().numpy())
            targets.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(targets)


def train_classifier(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, epochs: int, lr: float) -> RunHistory:
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = RunHistory(train_loss=[], val_accuracy=[])

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_items = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            total_items += xb.size(0)
        preds, targets = predict(model, val_loader)
        history.train_loss.append(total_loss / total_items)
        history.val_accuracy.append(accuracy_score(targets, preds))

    return history


## Step 2 - Pretrain on the source task

The source task is ordinary 10-class digit recognition. This gives the backbone a chance to learn general stroke and shape detectors.


In [ ]:
source_train_loader = make_loader(X_source_train, y_source_train, batch_size=64, shuffle=True)
source_test_loader = make_loader(X_source_test, y_source_test, batch_size=256)

source_model = FullModel(ConvBackbone(), num_classes=10).to(device)
source_history = train_classifier(source_model, source_train_loader, source_test_loader, epochs=10, lr=1e-3)
source_acc = source_history.val_accuracy[-1]
source_acc


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(source_history.train_loss, marker="o", label="train loss")
ax2 = ax.twinx()
ax2.plot(source_history.val_accuracy, marker="s", color="tab:green", label="val accuracy")
ax.set_xlabel("epoch")
ax.set_title("source-task training")
ax.set_ylabel("loss")
ax2.set_ylabel("accuracy")
plt.tight_layout()


## Step 3 - Build the target task

The target task asks whether a digit is prime or non-prime. We intentionally keep only a small labeled training set so that transfer learning has something to exploit.


In [ ]:
target_train_loader = make_loader(X_target_small, y_target_small, batch_size=32, shuffle=True)
target_test_loader = make_loader(X_source_test, y_target_test, batch_size=256)

np.bincount(y_target_small), np.bincount(y_target_test)


In [ ]:
# Frozen-feature transfer
transferred_backbone = copy.deepcopy(source_model.backbone)
for param in transferred_backbone.parameters():
    param.requires_grad = False

frozen_model = FullModel(transferred_backbone, num_classes=2).to(device)
frozen_history = train_classifier(frozen_model, target_train_loader, target_test_loader, epochs=12, lr=1e-3)

# Fine-tuning
finetune_model = FullModel(copy.deepcopy(source_model.backbone), num_classes=2).to(device)
finetune_history = train_classifier(finetune_model, target_train_loader, target_test_loader, epochs=12, lr=5e-4)

# Scratch baseline
scratch_model = FullModel(ConvBackbone(), num_classes=2).to(device)
scratch_history = train_classifier(scratch_model, target_train_loader, target_test_loader, epochs=12, lr=1e-3)


## Step 4 - Compare transfer strategies

Frozen transfer should help quickly if the source filters are already useful. Fine-tuning can do better when the target task differs enough to benefit from adaptation.


In [ ]:
results = {
    "frozen transfer": frozen_history.val_accuracy[-1],
    "fine-tune": finetune_history.val_accuracy[-1],
    "from scratch": scratch_history.val_accuracy[-1],
}
results


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, hist in [
    ("frozen transfer", frozen_history),
    ("fine-tune", finetune_history),
    ("from scratch", scratch_history),
]:
    ax.plot(hist.val_accuracy, marker="o", label=label)
ax.set_xlabel("epoch")
ax.set_ylabel("target-task accuracy")
ax.set_ylim(0.4, 1.0)
ax.set_title("prime vs non-prime transfer comparison")
ax.legend()
plt.tight_layout()


## Step 5 - Inspect reused filters and feature maps

Transfer learning is easier to trust when we look at what is being reused. The first-layer filters from the pretrained source model usually resemble edge or stroke detectors.


In [ ]:
pretrained_filters = source_model.backbone.net[0].weight.detach().cpu().numpy()
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, kernel in zip(axes.flat, pretrained_filters):
    ax.imshow(kernel[0], cmap="coolwarm")
    ax.axis("off")
plt.suptitle("pretrained conv1 filters")
plt.tight_layout()


In [ ]:
example_idx = 0
example_image = torch.tensor(X_source_test[example_idx : example_idx + 1]).to(device)
with torch.no_grad():
    feature_maps = source_model.backbone.net[0](example_image).cpu().numpy()[0]

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
axes[0, 0].imshow(X_source_test[example_idx, 0], cmap="gray")
axes[0, 0].set_title(f"digit={y_source_test[example_idx]}")
axes[0, 0].axis("off")
for ax, fmap in zip(axes.flat[1:], feature_maps):
    ax.imshow(fmap, cmap="magma")
    ax.axis("off")
plt.suptitle("source-model conv1 feature maps")
plt.tight_layout()


In [ ]:
final_preds, final_targets = predict(finetune_model, target_test_loader)
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(final_targets, final_preds, ax=ax, cmap="Greens", colorbar=False)
ax.set_title(f"fine-tuned target accuracy = {accuracy_score(final_targets, final_preds):.3f}")
plt.tight_layout()


## Exercises

- Make the target task even smaller and check when transfer becomes clearly better than training from scratch.
- Unfreeze only the second convolution block and compare that partial fine-tuning strategy with full fine-tuning.
- Replace the prime-vs-non-prime task with odd-vs-even or `digit < 5` and compare which labels transfer more easily.
